# 01 · Limpeza e qualidade dos dados

**Responsável:** Francisco Figueiredo  ·  **Frente de trabalho:** Preparação dos dados

---

### Objetivo:
Produzir o **dataset limpo que todas os outros vão usar**

### O que este notebook entrega ao resto do grupo:
O dataset limpo.


## 0 · Setup

Importar as bibliotecas.

In [67]:
import pandas as pd
import numpy as np

## 1 · Primeira leitura

Olhar para a tabela antes de lhe tocar: dimensões, nomes e tipos das colunas, primeiras linhas,
estatísticas descritivas das numéricas e das categóricas.


In [68]:
raw_df = pd.read_csv('../data/raw/artist_streams.csv', index_col = 0)
raw_df

raw_df.info()                    # tipos de cada coluna e quantos não-nulos
raw_df.describe()                # min, max, média 
raw_df.isna().sum()              # nulos por coluna
raw_df.duplicated().sum()        # linhas totalmente repetidas
raw_df['Country of Origin'].value_counts(dropna=False)   # categorias escritas de formas diferentes
raw_df.head()

<class 'pandas.DataFrame'>
Index: 511 entries, 0 to 500
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Artist Name            510 non-null    str    
 1   Sex                    486 non-null    str    
 2   Country of Origin      510 non-null    str    
 3   Primary Language       510 non-null    str    
 4   Primary Genre          510 non-null    str    
 5   Artist Type            510 non-null    str    
 6   Debut Year             510 non-null    float64
 7   Total Streams          505 non-null    float64
 8   Lead Streams           504 non-null    float64
 9   Feature Streams        499 non-null    float64
 10  Solo Streams           506 non-null    float64
 11  Collaborative Streams  509 non-null    float64
 12  Update Date            510 non-null    str    
 13  Year                   510 non-null    float64
dtypes: float64(7), str(7)
memory usage: 91.0 KB


,Artist Name,Sex,Country of Origin,Primary Language,Primary Genre,Artist Type,Debut Year,Total Streams,Lead Streams,Feature Streams,Solo Streams,Collaborative Streams,Update Date,Year
Artist ID,,,,,,,,,,,,,,
0,Drake,Male,Canada,English,Hip-Hop,Solo,2006.0,137492.1,94851.1,42640.9,53323.9,84168.2,2026-07-01 00:00:00,2026.0
1,Taylor Swift,Female,United States,English,Pop,Solo,2006.0,127861.0,126177.3,1683.8,113421.5,14439.5,2026-07-01 00:00:00,2026.0
2,Bad Bunny,Male,United States,Spanish,Reggaeton,Solo,2013.0,125899.8,82189.3,43710.5,48887.0,77012.8,2026-07-01 00:00:00,2026.0
3,The Weeknd,Male,Canada,English,R&B,Solo,2009.0,95703.2,77189.4,18513.8,51020.0,44683.2,2026-07-01 00:00:00,2026.0
4,Justin Bieber,Male,Canada,English,Pop,Solo,2009.0,78104.4,48316.2,29788.2,29638.3,48466.1,2026-07-01 00:00:00,2026.0


In [69]:
#PARA VER O DATASET INTEIRO:
#with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', None):
#    display(raw_df)

## 2 · Duplicados

### 2.1 Procurar duplicados e removê-los
Verificar linhas inteiramente repetidas, `Artist ID` repetidos, e nomes de artista repetidos com
identificadores diferentes.


In [70]:
print("Linhas inteiramente duplicadas:", raw_df.duplicated().sum())
print("Artistas com ID duplicado:", raw_df.index.duplicated().sum())
print("Nomes de artistas repetidos:", raw_df['Artist Name'].duplicated().sum())

# Mostrar as linhas que estão duplicadas
duplicadas = raw_df[raw_df.duplicated(keep=False)]
display(duplicadas)


Linhas inteiramente duplicadas: 10
Artistas com ID duplicado: 10
Nomes de artistas repetidos: 10


,Artist Name,Sex,Country of Origin,Primary Language,Primary Genre,Artist Type,Debut Year,Total Streams,Lead Streams,Feature Streams,Solo Streams,Collaborative Streams,Update Date,Year
Artist ID,,,,,,,,,,,,,,
45,Linkin Park,NaN,United States,English,Rock,Group,2000.0,33224.7,31426.6,1798.1,30151.2,3073.5,2026-07-01 00:00:00,2026.0
45,Linkin Park,NaN,United States,English,Rock,Group,2000.0,33224.7,31426.6,1798.1,30151.2,3073.5,2026-07-01 00:00:00,2026.0
140,Fleetwood Mac,NaN,United Kingdom,English,Rock,Group,1967.0,16577.5,16577.5,NaN,16567.4,10.1,2026-07-01 00:00:00,2026.0
140,Fleetwood Mac,NaN,United Kingdom,English,Rock,Group,1967.0,16577.5,16577.5,NaN,16567.4,10.1,2026-07-01 00:00:00,2026.0
218,John Mayer,Male,United States,English,Pop,Solo,1999.0,13208.3,12134.4,1076.8,11821.4,1386.9,2026-07-01 00:00:00,2026.0
218,John Mayer,Male,United States,English,Pop,Solo,1999.0,13208.3,12134.4,1076.8,11821.4,1386.9,2026-07-01 00:00:00,2026.0
221,System Of A Down,Male,United States,English,Metal,Group,1998.0,13009.4,13017.1,0.0,12963.5,45.9,2026-07-01 00:00:00,2026.0
221,System Of A Down,Male,United States,English,Metal,Group,1998.0,13009.4,13017.1,0.0,12963.5,45.9,2026-07-01 00:00:00,2026.0
263,Zedd,Male,Russia,English,EDM,Solo,2010.0,11481.4,11486.5,0.0,235.7,11245.7,2026-07-01 00:00:00,2026.0


In [71]:
# Verificar se há nomes de artistas repetidos com update date diferentes
dup_nomes = raw_df[raw_df['Artist Name'].notna() & raw_df.duplicated(subset=['Artist Name'], keep=False)].copy()
dup_nomes['Update Date_dt'] = pd.to_datetime(dup_nomes['Update Date'], errors='coerce')
print("\nArtistas repetidos ordenados por nome e data:")
print(dup_nomes[['Artist Name', 'Total Streams', 'Update Date', 'Year']].sort_values(by=['Artist Name', 'Update Date']))
#As datas não divergem, são todas cópias exatas

#Decisão: Remover cópias exatas e guardar num novo DataFrame de trabalho
df = raw_df.drop_duplicates().copy()
print(f"\nDimensões após remoção de duplicados exatos: {df.shape}")



Artistas repetidos ordenados por nome e data:
                  Artist Name  Total Streams          Update Date    Year
Artist ID                                                                
307                 Burna Boy        10290.6  2026-07-01 00:00:00  2026.0
307                 Burna Boy        10290.6  2026-07-01 00:00:00  2026.0
275        Christina Aguilera        11060.5  2026-07-01 00:00:00  2026.0
275        Christina Aguilera        11060.5  2026-07-01 00:00:00  2026.0
272                 DJ Khaled        11196.0  2026-07-01 00:00:00  2026.0
272                 DJ Khaled        11196.0  2026-07-01 00:00:00  2026.0
441                Disclosure         7787.8  2026-07-01 00:00:00  2026.0
441                Disclosure         7787.8  2026-07-01 00:00:00  2026.0
140             Fleetwood Mac        16577.5  2026-07-01 00:00:00  2026.0
140             Fleetwood Mac        16577.5  2026-07-01 00:00:00  2026.0
218                John Mayer        13208.3  2026-07-01 00:00:00

## 3 · Missing Values

### 3.1 Mapear os NaN
Quantos valores em falta tem cada coluna, em número absoluto.

### 3.2 Decidir coluna a coluna
A decisão depende da coluna e da proporção: apagar linhas, preencher, ou deixar em falta e assinalar.
Uma coluna com poucos nulos não se trata como uma com metade dos valores em falta.


In [72]:
#Valores em falta
nulos = pd.DataFrame({"Nulos": df.isnull().sum()})
print(nulos)

#Preencher nulos em 'Sex' para bandas/grupos onde não se aplica
df1 = df.copy()
df1 = df1.dropna(how='all', axis=0)
df1['Sex'] = df1['Sex'].fillna('Group / NaN')
#Remover linha totalmente vazia (ex: Artist ID 500) que está vazia
df1 = df1.dropna(subset=['Artist Name'], axis=0)



                       Nulos
Artist Name                1
Sex                       23
Country of Origin          1
Primary Language           1
Primary Genre              1
Artist Type                1
Debut Year                 1
Total Streams              6
Lead Streams               7
Feature Streams           10
Solo Streams               5
Collaborative Streams      2
Update Date                1
Year                       1


In [73]:
# Para colunas essenciais com poucos nulos (ex: Artist Name, Country)
print(f"Linhas com nulos restantes: {df1[df1.isnull().any(axis=1)].shape[0]}")
print()

# Calcular valores em falta usando a relação Total = Solo + Collaborative
mask_total = df1['Total Streams'].isna() & df1['Solo Streams'].notna() & df1['Collaborative Streams'].notna()
df1.loc[mask_total, 'Total Streams'] = (df1.loc[mask_total, 'Solo Streams'] + df1.loc[mask_total, 'Collaborative Streams']).round(1)

mask_solo = df1['Solo Streams'].isna() & df1['Total Streams'].notna() & df1['Collaborative Streams'].notna()
df1.loc[mask_solo, 'Solo Streams'] = (df1.loc[mask_solo, 'Total Streams'] - df1.loc[mask_solo, 'Collaborative Streams']).round(1)

mask_collab = df1['Collaborative Streams'].isna() & df1['Total Streams'].notna() & df1['Solo Streams'].notna()
df1.loc[mask_collab, 'Collaborative Streams'] = (df1.loc[mask_collab, 'Total Streams'] - df1.loc[mask_collab, 'Solo Streams']).round(1)

print("Nulos restantes após preenchimento por decomposição Solo/Collab:")
print(df1[['Total Streams', 'Solo Streams', 'Collaborative Streams']].isna().sum())

Linhas com nulos restantes: 25

Nulos restantes após preenchimento por decomposição Solo/Collab:
Total Streams            0
Solo Streams             0
Collaborative Streams    0
dtype: int64


In [74]:
# Preencher Lead Streams em falta
mask_lead = df1['Lead Streams'].isna() & df1['Total Streams'].notna() & df1['Feature Streams'].notna()
df1.loc[mask_lead, 'Lead Streams'] = (df1.loc[mask_lead, 'Total Streams'] - df1.loc[mask_lead, 'Feature Streams']).clip(lower=0)

# Preencher Feature Streams em falta
mask_feat = df1['Feature Streams'].isna() & df1['Total Streams'].notna() & df1['Lead Streams'].notna()
df1.loc[mask_feat, 'Feature Streams'] = (df1.loc[mask_feat, 'Total Streams'] - df1.loc[mask_feat, 'Lead Streams']).clip(lower=0)

print(df1[['Lead Streams', 'Feature Streams']].isna().sum())

Lead Streams       0
Feature Streams    0
dtype: int64


## 4 · Outros problemas

### 4.1 Coerência das duas decomposições de streams
O guião garante que `Total Streams = Solo + Collaborative` e `Total Streams = Lead + Feature`.
Temos de verificar as duas instâncias e investigar as linhas em que não se confirmam.

### 4.2 Categorias escritas de várias maneiras
`Country of Origin`, `Primary Language`, `Primary Genre`, `Artist Type` e `Sex`: listar os valores
distintos de cada uma e procurar o mesmo valor escrito de formas diferentes.

In [75]:
# 4.1 · Coerência das decomposições de streams
#Total = Solo + Collaborative
coerente_sc = np.isclose(df1['Total Streams'], df1['Solo Streams'] + df1['Collaborative Streams'], equal_nan=True)
print(f"Inconsistências em Solo/Collab: {(~coerente_sc).sum()}")

#Total = Lead + Feature
coerente_lf = np.isclose(df1['Total Streams'], df1['Lead Streams'] + df1['Feature Streams'], equal_nan=True)
print(f"Inconsistências em Lead/Feature: {(~coerente_lf).sum()}")


# 4.2 · Detetar inconsistências de escrita em colunas categóricas (strings)
colunas_string = ['Artist Name', 'Sex', 'Country of Origin', 'Primary Language', 'Primary Genre', 'Artist Type']

for col in colunas_string:
    df1[col] = df1[col].apply(lambda x: x.strip() if isinstance(x, str) else x)

colunas_categoricas = ['Sex', 'Country of Origin', 'Primary Language', 'Primary Genre', 'Artist Type']

for col in colunas_categoricas:
    print()
    print(f"{col} (valores únicos: {df1[col].nunique(dropna=False)})")
    print(df1[col].value_counts(dropna=False).to_string())
#Conclusão: Está tudo correto

Inconsistências em Solo/Collab: 0
Inconsistências em Lead/Feature: 401

Sex (valores únicos: 3)
Sex
Male           390
Female          88
Group / NaN     22

Country of Origin (valores únicos: 43)
Country of Origin
United States          265
United Kingdom          55
Mexico                  27
Canada                  18
Colombia                16
Brazil                  15
India                   13
South Korea             11
Argentina                9
Australia                7
Germany                  7
Spain                    6
France                   5
Sweden                   5
Ireland                  4
Italy                    3
Scotland                 2
Norway                   2
Netherlands              2
Jamaica                  2
Japan                    2
Nigeria                  2
Dominican Republic       2
Barbados                 1
Trinidad and Tobago      1
Cuba                     1
Panama                   1
Senegal                  1
New Zealand              1
Mo

## 5 · Conversões de tipo

Converter o que ficou com o tipo errado.


In [76]:
df1['Debut Year'] = df1['Debut Year'].astype('Int64')  #Converter anos para inteiro 
df1['Year'] = df1['Year'].astype('Int64')

df1['Update Date'] = pd.to_datetime(df1['Update Date']) #Converter Update Date para o tipo datetime

stream_cols = ['Total Streams', 'Lead Streams', 'Feature Streams', 'Solo Streams', 'Collaborative Streams']

#Converter de milhões para int real completo 
df1[stream_cols] = (df1[stream_cols] * 1_000_000).round().astype('Int64')

print("Tipos de dados atualizados:")
print(df1.dtypes)

Tipos de dados atualizados:
Artist Name                         str
Sex                                 str
Country of Origin                   str
Primary Language                    str
Primary Genre                       str
Artist Type                         str
Debut Year                        Int64
Total Streams                     Int64
Lead Streams                      Int64
Feature Streams                   Int64
Solo Streams                      Int64
Collaborative Streams             Int64
Update Date              datetime64[us]
Year                              Int64
dtype: object


## 6 · Datas de atualização

### 6.1 (3a) Atualização mais recente e mais antiga

### 6.2 (3b) Registos desatualizados
Que artistas têm a última atualização **anterior ao fim de 2025** e, por isso, precisam de ver as
contagens atualizadas.


In [77]:
#Ver se a Update Date está atualizada
datas_dt = pd.to_datetime(df1['Update Date'], errors='coerce')
datas_validas = datas_dt.dropna()

print("Atualização mais antiga:", datas_validas.min().strftime('%Y-%m-%d'))
print("Atualização mais recente:", datas_validas.max().strftime('%Y-%m-%d'))

print("\nContagem por ano de atualização:")
print(datas_validas.dt.year.value_counts().sort_index())

# Artistas com Update Date desatualizada
desatualizados = df1[datas_dt < pd.Timestamp('2026-01-01')].copy()
print("\nArtistas desatualizados:")
print(desatualizados[['Artist Name', 'Update Date']].sort_values(by='Update Date').to_string(index=False))

Atualização mais antiga: 2025-07-01
Atualização mais recente: 2026-07-01

Contagem por ano de atualização:
Update Date
2025     10
2026    490
Name: count, dtype: int64

Artistas desatualizados:
        Artist Name Update Date
              Quavo  2025-07-01
         Gucci Mane  2025-07-01
     Grupo Frontera  2025-07-01
      Elvis Presley  2025-07-01
        Teddy Swims  2025-07-01
Alejandro Fernández  2025-07-01
          SEVENTEEN  2025-07-01
        ScHoolboy Q  2025-07-01
         Kane Brown  2025-07-01
 Grupo Menos É Mais  2025-07-01


## 7 · Exportar e documentar

### 7.1 Gravar o dataset limpo
Exportar para `data/processed/artistas_limpo.csv`.


In [78]:
print("Dimensões finais de df1:", df1.shape)
print("Total de nulos restantes por coluna:\n", df1.isna().sum())

df1.to_csv('../data/processed/artistas_limpo.csv', index=True)

Dimensões finais de df1: (500, 14)
Total de nulos restantes por coluna:
 Artist Name              0
Sex                      0
Country of Origin        0
Primary Language         0
Primary Genre            0
Artist Type              0
Debut Year               0
Total Streams            0
Lead Streams             0
Feature Streams          0
Solo Streams             0
Collaborative Streams    0
Update Date              0
Year                     0
dtype: int64
